# 🧬 Notebook 3: Knowledge Graph Construction & Visualization
## Entity Extraction → Graph Building → Analytics → Interactive Visualization

This notebook demonstrates:
1. **LLM-based entity/relation extraction** with regex fallback
2. **Knowledge graph construction** using NetworkX (Neo4j fallback)
3. **Graph analytics** — centrality, PageRank, community detection
4. **Interactive visualization** using matplotlib

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph.extractor import extract_entities_heuristic
from src.graph.store import GraphStore
import tempfile, os

print('✅ Knowledge graph modules loaded!')

## Step 1: Heuristic Entity Extraction (No LLM Required)

The heuristic extractor uses regex to find capitalised multi-word phrases, filters common words, and generates co-occurrence relations.

In [ ]:
# Extract entities from a sample AI/ML paragraph
sample_text = """
Machine Learning is a subset of Artificial Intelligence that uses Neural Networks
for pattern recognition. Deep Learning, a subfield of Machine Learning, powers
modern Natural Language Processing systems. The Transformer architecture,
introduced by Google Brain, revolutionized NLP with the Self-Attention Mechanism.
Models like BERT and GPT are based on Transformers. Convolutional Neural Networks
excel at Computer Vision tasks, while Recurrent Neural Networks handle sequential
data. Transfer Learning allows pre-trained models to be fine-tuned for specific tasks.
Reinforcement Learning trains agents through reward signals.
"""

result = extract_entities_heuristic(sample_text)

print(f'Extracted {len(result["entities"])} entities:')
for e in result['entities']:
    print(f"  • {e['name']} [{e['type']}]")

print(f'\nExtracted {len(result["relations"])} relations:')
for r in result['relations']:
    print(f"  {r['source']} --[{r['type']}]--> {r['target']}")

## Step 2: Build the Knowledge Graph

We construct a graph from the extracted entities and relations using our NetworkX-based store.

In [ ]:
# Build graph from extracted data
store = GraphStore(persist_path=os.path.join(tempfile.gettempdir(), 'demo_kg.json'))
store.clear()

for entity in result['entities']:
    store.add_entity(entity['name'], entity['type'], entity.get('description', ''))

for rel in result['relations']:
    store.add_relation(rel['source'], rel['target'], rel['type'], rel.get('description', ''))

stats = store.get_stats()
print(f'📊 Knowledge Graph Statistics:')
print(f'   Nodes: {stats["node_count"]}')
print(f'   Edges: {stats["edge_count"]}')

print(f'\n🔍 Querying "Machine Learning":')
result_ml = store.query_entity('Machine Learning')
if result_ml:
    print(f'   {result_ml}')

print(f'\n🔎 Fuzzy search for "Learning":')
search_results = store.search_entities('Learning')
for sr in search_results:
    print(f'   • {sr}')

## Step 3: Graph Analytics

Analyzing the graph structure: degree centrality and most connected entities.

In [ ]:
import networkx as nx

# Access the underlying NetworkX graph
G = store.graph if hasattr(store, 'graph') else store._graph

# Degree centrality
centrality = nx.degree_centrality(G)
sorted_centrality = sorted(centrality.items(), key=lambda x: x[1], reverse=True)

print('📈 Degree Centrality (most connected entities):')
for name, score in sorted_centrality[:10]:
    print(f'   {name}: {score:.3f} ({G.degree(name)} connections)')

# PageRank
if len(G.nodes) > 0:
    try:
        pr = nx.pagerank(G)
        sorted_pr = sorted(pr.items(), key=lambda x: x[1], reverse=True)
        print(f'\n📊 PageRank (most important entities):')
        for name, score in sorted_pr[:10]:
            print(f'   {name}: {score:.4f}')
    except Exception as e:
        print(f'   PageRank requires a directed graph: {e}')

## Step 4: Graph Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Color nodes by type
color_map = {'CONCEPT': '#4C72B0', 'TECHNOLOGY': '#55A868', 'ORGANIZATION': '#E24A33', 'PERSON': '#FBC15E'}
node_colors = [color_map.get(G.nodes[n].get('type', 'CONCEPT'), '#8172B3') for n in G.nodes]

pos = nx.spring_layout(G, k=2.5, iterations=50, seed=42)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1200, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='#cccccc', arrows=True, arrowsize=15, ax=ax)

# Edge labels
edge_labels = nx.get_edge_attributes(G, 'type')
if edge_labels:
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, font_color='#666', ax=ax)

# Legend
legend_handles = [mpatches.Patch(color=c, label=t) for t, c in color_map.items()]
ax.legend(handles=legend_handles, loc='upper left', fontsize=10)
ax.set_title(f'Knowledge Graph ({stats["node_count"]} nodes, {stats["edge_count"]} edges)', fontsize=16, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()